In [1]:
# how to uncover truths that don't matter - third section

import time

import numpy as np
import coriolis_functions 

import matplotlib.pyplot as plt 
import seaborn as sns

import pandas as pd

# loading dataframe with airport information
ldf = pd.read_csv("data/airports.csv").astype(
    {
    "IATA": "category", 
    "AIRPORT": "category", 
    "CITY": "category", 
    "STATE": "category", 
    "COUNTRY": "category", 
    }
)

# loading dataframe with not cancelled flights
fdf = pd.read_csv("data/not_cancelled_flights.csv").astype(
    {
     "FL_DATE": "datetime64[ns]", 
     "AIRLINE": "category", 
     "AIRLINE_DOT": "category", 
     "AIRLINE_CODE": "category", 
     "ORIGIN_CITY": "category", 
     "DEST_CITY": "category", 
     "CANCELLATION_CODE": "category", 
     "DIVERTED": "bool",
     "arr_datetime": "datetime64[ns]",
     "dep_datetime": "datetime64[ns]", 
     "crs_arr_datetime": "datetime64[ns]", 
     "crs_dep_datetime": "datetime64[ns]", 
     "woff_datetime": "datetime64[ns]",
     "won_datetime": "datetime64[ns]", 
     "ORIGIN": "category", 
     "DEST": "category",
    }
)

In [2]:
# Merge fdf with ldf to add geographical data for ORIGIN and DEST
fdf_merged = pd.merge(fdf, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="ORIGIN", right_on="IATA", how="left")
fdf_merged = pd.merge(fdf_merged, ldf[["IATA", "LATITUDE", "LONGITUDE"]], 
                      left_on="DEST", right_on="IATA", how="left", suffixes=("_ORIGIN", "_DEST"))

# Drop original origin and destination columns
fdf_merged = fdf_merged.drop(["ORIGIN", "DEST"], axis=1)

In [3]:
start = time.time()

In [ ]:
# Apply direction vector function row-wise
directions = fdf_merged.apply(
    lambda row: coriolis_functions.direction_vector(
        row["LATITUDE_ORIGIN"], 
        row["LONGITUDE_ORIGIN"], 
        row["LATITUDE_DEST"], 
        row["LONGITUDE_DEST"]
    ), axis=1
)

In [ ]:
end = time.time()

In [ ]:
print(f"{end - start} - calculationtime of direction_vector written in python [s]")

In [ ]:
# Split the resulting direction vectors into separate columns
fdf_merged["x_direction"] = directions.apply(lambda x: x[0])
fdf_merged["y_direction"] = directions.apply(lambda x: x[1])
fdf_merged["z_direction"] = directions.apply(lambda x: x[2])

In [ ]:
# division of drift through distance gives percentage
fdf_merged.loc[:, "drift_factor"] = fdf_merged["total_drift_distance"] / fdf_merged["haversine_distance"].replace(0, pd.NA)

In [ ]:
ldf.dtypes

In [ ]:
fdf.dtypes

In [ ]:
fdf_merged.dtypes